**DF_matches**

In [0]:
# 1. Importation des fonctions de Databricks / PySpark
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

# 1. Définition de la structure des données
schema = StructType([
    StructField("match_id", IntegerType(), True),
    StructField("home_team", StringType(), True),
    StructField("away_team", StringType(), True),
    StructField("home_goals", IntegerType(), True),
    StructField("away_goals", IntegerType(), True)
])

# 2. Simulation de 24 matchs impliquant de nombreuses équipes européennes
donnees_champions_league = [
    (1, "Paris SG", "Real Madrid", 3, 1),
    (2, "Bayern Munich", "Arsenal", 2, 2),
    (3, "FC Barcelone", "Monaco", 1, 2),
    (4, "Manchester City", "Juventus", 4, 0),
    (5, "Inter Milan", "Liverpool", 0, 2),
    (6, "Bayer Leverkusen", "AC Milan", 2, 1),
    (7, "Atletico Madrid", "RB Leipzig", 1, 0),
    (8, "Borussia Dortmund", "Sporting CP", 3, 2),
    (9, "Benfica", "Feyenoord", 1, 1),
    (10, "PSV Eindhoven", "Girona", 2, 0),
    (11, "Aston Villa", "Bologna", 2, 1),
    (12, "Celtic Glasgow", "Club Brugge", 0, 0),
    (13, "Stuttgart", "Lille OSC", 1, 2),
    (14, "Eintracht Frankfurt", "Lazio", 2, 3),
    (15, "Real Sociedad", "AS Roma", 1, 1),
    (16, "Shakhtar Donetsk", "Dinamo Zagreb", 0, 2),
    (17, "Salzburg", "Brest", 1, 3),
    (18, "Sparta Prague", "Young Boys", 2, 0),
    (19, "Sturm Graz", "Sporting CP", 0, 2),
    (20, "Crvena Zvezda", "Monaco", 1, 4),
    (21, "Real Madrid", "Borussia Dortmund", 5, 2),
    (22, "Paris SG", "PSV Eindhoven", 1, 1),
    (23, "Juventus", "VfB Stuttgart", 0, 1),
    (24, "Arsenal", "Shakhtar Donetsk", 1, 0)
]

# 3. Création du DataFrame global
df_matches = spark.createDataFrame(donnees_champions_league, schema)

# 4. Calcul des points à domicile et à l'extérieur
df_clean = df_matches.withColumn(
    "home_points", 
    F.when(F.col("home_goals") > F.col("away_goals"), 3)
     .when(F.col("home_goals") == F.col("away_goals"), 1)
     .otherwise(0)
).withColumn(
    "away_points",
    F.when(F.col("away_goals") > F.col("home_goals"), 3)
     .when(F.col("away_goals") == F.col("home_goals"), 1)
     .otherwise(0)
)

print("Nouveau jeu de données de la Ligue des Champions chargé avec succès !")



**DATAFRAMES**

In [0]:
# 1. Extraction des statistiques pour les équipes à domicile
df_home = df_clean.select(
    F.col("home_team").alias("team"),
    F.col("home_points").alias("points"),
    F.col("home_goals").alias("goals_scored"),
    F.col("away_goals").alias("goals_conceded")
)

# 2. Extraction des statistiques pour les équipes à l'extérieur
df_away = df_clean.select(
    F.col("away_team").alias("team"),
    F.col("away_points").alias("points"),
    F.col("away_goals").alias("goals_scored"),
    F.col("home_goals").alias("goals_conceded")
)

# 3. Fusion des deux tableaux (Domicile + Extérieur)
df_all_matches = df_home.union(df_away)

# 4. Calcul du classement agrégé par équipe
df_ranking = df_all_matches.groupBy("team").agg(
    F.count("points").alias("matches_played"),
    F.sum("points").alias("total_points"),
    F.sum("goals_scored").alias("total_scored"),
    F.sum("goals_conceded").alias("total_conceded")
)

# 5. Ajout de la différence de buts et tri selon les règles de l'UEFA
df_final_ladder = df_ranking.withColumn(
    "goal_difference", 
    F.col("total_scored") - F.col("total_conceded")
).orderBy(
    F.col("total_points").desc(),      # Équipe avec le plus de points en premier
    F.col("goal_difference").desc(),   # En cas d'égalité : meilleure différence de buts
    F.col("total_scored").desc()       # En cas d'égalité secondaire : meilleure attaque
)

# 6. Affichage du classement final
display(df_final_ladder)


**Identifier les Qualifiés, Barragistes et Éliminés**

In [0]:
from pyspark.sql.window import Window

# 1. Générer la position exacte (Rang de 1 à 36)
windowSpec = Window.orderBy(
    F.col("total_points").desc(), 
    F.col("goal_difference").desc(), 
    F.col("total_scored").desc()
)

df_with_rank = df_final_ladder.withColumn("rank", F.row_number().over(windowSpec))

# 2. Ajouter le statut UEFA officiel selon la position
df_uefa_status = df_with_rank.withColumn(
    "uefa_status",
    F.when(F.col("rank") <= 8, "🟢 Qualifié (Top 8)")
     .when(F.col("rank") <= 24, "🟡 Barragiste (9-24)")
     .otherwise("🔴 Éliminé (25-36)")
)

# 3. Affichage final
display(df_uefa_status.select("rank", "team", "total_points", "goal_difference", "total_scored", "uefa_status"))


Databricks visualization. Run in Databricks to view.